In [1]:
%%capture output
#install Wikipedia API
!pip3 install wikipedia-api


In [2]:
import numpy as np
import pandas as pd
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

pd.options.display.max_columns = 100

import nltk
# nltk.download('omw-1.4')
nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

import wikipediaapi


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
%%capture
!python -m textblob.download_corpora


In [4]:
sentence_1 = 'Jen is a good student.'
sentence_2 = 'Jen is also a great guitarist.'
sentence_3 = 'Good students can sometimes be good guitarists'


# Data Cleaning
We want to singularize guitarists and students.

In [5]:
sentence_3_tb = TextBlob(sentence_3) # Make a textblob so that we can singularize the word
sentence_3_singular = [x.singularize() for x in sentence_3_tb.words] # Singularize each word in the text
sentence_3_clean = ' '.join(sentence_3_singular) # Join it together into a single string
sentence_3_clean


'Good student can sometime be good guitarist'

## Bag of Words Using CountVectorizer

In [6]:
# Perform the count transformation
vectorizer = CountVectorizer(stop_words='english')
bow_vec = vectorizer.fit_transform([sentence_1, sentence_2, sentence_3_clean])
bow_vec


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 9 stored elements and shape (3, 5)>

In [7]:
bow_vec.toarray()


array([[1, 0, 0, 1, 1],
       [0, 1, 1, 1, 0],
       [2, 0, 1, 0, 1]])

In [8]:
# Print out results in a data frame
sent_df = pd.DataFrame(bow_vec.toarray(), columns = vectorizer.get_feature_names_out())
sent_df


,good,great,guitarist,jen,student
0,1,0,0,1,1
1,0,1,1,1,0
2,2,0,1,0,1


In [9]:
[sentence_1, sentence_2, sentence_3_clean]


['Jen is a good student.',
 'Jen is also a great guitarist.',
 'Good student can sometime be good guitarist']

### Your Turn
1. Write 4 sentences of your choice.
2. Run the `CountVectorizer` on your sentences.
3. Print the results in a data frame.

In [14]:
# Solution 1
my_sents = [
    "It was the best of times, it was the worst of times.",
    "Call me Ishmael.",
    "To be, or not to be.  That is the question.",
    "We do not choose these things because they are easy."
]
my_sents

['It was the best of times, it was the worst of times.',
 'Call me Ishmael.',
 'To be, or not to be.  That is the question.',
 'We do not choose these things because they are easy.']

In [15]:
# Solution 2
my_vectorizer = CountVectorizer(stop_words='english')
my_bow_vec = my_vectorizer.fit_transform( my_sents )
my_bow_vec.toarray()


array([[1, 0, 0, 0, 0, 0, 2, 1],
       [0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0],
       [0, 1, 1, 0, 0, 1, 0, 0]])

In [16]:
# Solution 3
my_sent_df = pd.DataFrame(my_bow_vec.toarray(), columns = my_vectorizer.get_feature_names_out())
my_sent_df


,best,choose,easy,ishmael,question,things,times,worst
0,1,0,0,0,0,0,2,1
1,0,0,0,1,0,0,0,0
2,0,0,0,0,1,0,0,0
3,0,1,1,0,0,1,0,0


## TF-IDF

In [17]:
# Perform the TF-IDF transformation - Option 1 (TfidfVectorizer)
tf_idf_vec = TfidfVectorizer(stop_words = 'english')
tf_idf_jen = tf_idf_vec.fit_transform([sentence_1, sentence_2, sentence_3_clean])
tf_idf_jen


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>

In [18]:
print(sentence_1)
print(sentence_2)
print(sentence_3_clean)


Jen is a good student.
Jen is also a great guitarist.
Good student can sometime be good guitarist


In [19]:
# Print out results in a dataframe
tf_df = pd.DataFrame(tf_idf_jen.toarray(), columns = tf_idf_vec.get_feature_names_out())
tf_df.shape


(3, 5)

In [20]:
tf_df


,good,great,guitarist,jen,student
0,0.577350,0.000000,0.000000,0.577350,0.577350
1,0.000000,0.680919,0.517856,0.517856,0.000000
2,0.816497,0.000000,0.408248,0.000000,0.408248


In [21]:
# Perform the TF-IDF transformation - Option 2 (CountVectorizer + TfidfTransformer - better for large datasets)
tf_idf_tran = TfidfTransformer()
tf_idf_jen = tf_idf_tran.fit_transform(bow_vec)
tf_idf_jen

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>

In [22]:
# Print out results in a dataframe
tf_df = pd.DataFrame(tf_idf_jen.toarray(), columns = vectorizer.get_feature_names_out())
tf_df


,good,great,guitarist,jen,student
0,0.577350,0.000000,0.000000,0.577350,0.577350
1,0.000000,0.680919,0.517856,0.517856,0.000000
2,0.816497,0.000000,0.408248,0.000000,0.408248


In [23]:
# Get a data frame with the TF-IDF values sorted for document 0
df = pd.DataFrame(tf_idf_jen[0].T.todense(), index=tf_idf_vec.get_feature_names_out(), columns=["TF-IDF"])
df = df.sort_values('TF-IDF', ascending=False)
df


,TF-IDF
good,0.57735
student,0.57735
jen,0.57735
guitarist,0.00000
great,0.00000


In [24]:
tf_df.transpose()[0].sort_values(ascending = False)

,0
good,0.57735
student,0.57735
jen,0.57735
guitarist,0.00000
great,0.00000


### Your Turn
1. Use the `TfidfTransformer` to transform the bag of words matrix you created above to TF-IDF.
2. Print out the results in a data frame.

These settings might be helpful.

In [29]:
np.set_printoptions(linewidth=200)
np.set_printoptions(precision=6, formatter={'all': lambda x: f'{x:.4f}'})
my_bow_vec.toarray()

array([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 2.0000, 1.0000],
       [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 1.0000, 1.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000]])

In [30]:
# Solution 1
my_tf_idf_tran = TfidfTransformer()
my_tf_idf_jen = my_tf_idf_tran.fit_transform(my_bow_vec)
my_tf_idf_jen


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8 stored elements and shape (4, 8)>

In [31]:
my_tf_idf_jen.toarray()


array([[0.4082, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.8165, 0.4082],
       [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 0.5774, 0.5774, 0.0000, 0.0000, 0.5774, 0.0000, 0.0000]])

In [32]:
# Solution 2
my_tf_df = pd.DataFrame(my_tf_idf_jen.toarray(), columns = my_vectorizer.get_feature_names_out())
my_tf_df


,best,choose,easy,ishmael,question,things,times,worst
0,0.408248,0.00000,0.00000,0.0,0.0,0.00000,0.816497,0.408248
1,0.000000,0.00000,0.00000,1.0,0.0,0.00000,0.000000,0.000000
2,0.000000,0.00000,0.00000,0.0,1.0,0.00000,0.000000,0.000000
3,0.000000,0.57735,0.57735,0.0,0.0,0.57735,0.000000,0.000000


# === Start Here ===


# Another Example - Using Wikipedia API

In [ ]:
# Pull out the popcorn page from wikipedia - https://en.wikipedia.org/wiki/Popcorn
topic = 'popcorn'
wikip = wikipediaapi.Wikipedia(user_agent = 'foobar')
page_ex = wikip.page(topic)
wiki_text = page_ex.text
wiki_text


In [ ]:
print(wiki_text[:4000])


### Clean the text - version 1

Using string replace.



In [ ]:
# Replace newline chars with spaces before doing any processing. Strip the ' and "s" from possessives
wiki_text_clean = (
    wiki_text
    .lower()
    .replace("\n"," ")
    .replace("'s",'')
    .replace("'",'')
)
wiki_text_clean


In [ ]:
print(wiki_text_clean[:4000])


### Clean the text - version 2

Using a for..loop and string replace.




In [ ]:
wiki_text_clean = wiki_text.lower()
for c in ["\n", "'s", "'", "  "]:
  wiki_text_clean = wiki_text_clean.replace(c," ")
wiki_text_clean


### Clean the text - version 3

Using a regular expression.


In [ ]:
import re

pat = re.compile(r"(\n|'s|'| )+")
wiki_text_clean = re.sub(pat,' ', wiki_text.lower())
wiki_text_clean


### Moving on ...


In [ ]:
# Break up single string into separate sentences
wiki_blob = TextBlob(wiki_text_clean)
len(wiki_blob.sentences)


In [ ]:
# Only look at first 20 sentences
my_sentences = wiki_blob.sentences
my_sentences


In [ ]:
# Convert text blob sentences to strings
my_sentences_str = [ str(x) for x in my_sentences ]
my_sentences_str


In [ ]:
# Perform the TF-IDF Vectorization
tf_idf_vec = TfidfVectorizer(stop_words = 'english')
tf_idf_pop = tf_idf_vec.fit_transform(my_sentences_str)
tf_idf_pop.shape


In [ ]:
128*878

In [ ]:
tf_idf_pop


In [ ]:
tf_idf_pop.transpose().shape


In [ ]:
np.set_printoptions(precision=6, formatter=None)
tf_idf_vec.get_feature_names_out()


In [ ]:
for w in ['use', 'used', 'uses', 'using',]:
  print(f"stem: {w} -> {TextBlob( w ).words[0].stem()}")

for w in ['use', 'used', 'uses', 'using',]:
  print(f"lemmatize: {w} -> {TextBlob( w ).words[0].lemmatize()}")

for w in ['use', 'used', 'uses', 'using',]:
  print(f"singularize: {w} -> {TextBlob( w ).words[0].singularize()}")


In [ ]:
# Print out results in a dataframe
tf_df = pd.DataFrame(tf_idf_pop.toarray(), columns = tf_idf_vec.get_feature_names_out())
tf_df.transpose()


In [ ]:
# Get a data frame with the TF-IDF values sorted for document 0
df = pd.DataFrame(tf_idf_pop[0].T.todense(), index=tf_idf_vec.get_feature_names_out(), columns=["TF-IDF"])
df = df.sort_values('TF-IDF', ascending=False)
df[:5]


In [ ]:
tf_df.T[[0]].sort_values([0], ascending=False)[:5]


In [ ]:
(
  tf_idf_pop[0]
  .T
  .todense()
)
